# Accurate parser check on a Colab T4

Starts the accurate parser (MinerU 4) on this runtime's GPU and runs the same checks the
container tests run on CPU. Every check prints PASS or FAIL, and the last cell fails the
notebook if any check failed.

**Before you start:** Runtime > Change runtime type > T4 GPU. This does not test the Docker
image (Colab has no Docker daemon); it tests the Python stack the image runs: MinerU, the
service and the pinned models, on the GPU the image targets.

Set `BRANCH` below to the branch or tag to test.

In [ ]:
BRANCH = "main"          # the branch or tag to test
REPO = "https://github.com/daddal001/two-tier-document-parser.git"
TIER = "basic"           # "basic" = pipeline models on the GPU; "standard" also needs vLLM (see the last section)

## 1. The runtime

In [ ]:
import subprocess, sys

def sh(command, check=True):
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    if check and result.returncode:
        raise RuntimeError(f"{command}\n{result.stdout}\n{result.stderr}")
    return (result.stdout + result.stderr).strip()

gpu = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False)
print("GPU:", gpu or "NONE FOUND - change the runtime type to T4 GPU")
print("Notebook Python:", sys.version.split()[0], "(the service gets its own Python 3.12 below)")

## 2. Get the code and a Python 3.12 environment

In [ ]:
import os
WORK = "/content/work"
sh(f"rm -rf {WORK} && mkdir -p {WORK}")
print(sh(f"git clone --depth 1 --branch {BRANCH} --recurse-submodules --shallow-submodules {REPO} {WORK}/repo"))
sh("pip install -q uv")
print(sh(f"uv venv --python 3.12 {WORK}/venv"))
PY = f"{WORK}/venv/bin/python"
print(sh(f"{PY} --version"))

## 3. Install MinerU and the service

Uses the repository's pinned dependency limits (`deploy/constraints/mineru-compat.txt`). This
takes a few minutes.

In [ ]:
REPO_DIR = f"{WORK}/repo"
install = (
    f"cd {REPO_DIR} && uv pip install --python {PY} "
    "-c deploy/constraints/mineru-compat.txt "
    "-e './MinerU[torch]' -e '.[accurate,telemetry]' pypdf"
)
print(sh(install)[-600:])
print(sh(f"{PY} -c \"import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())\""))

## 4. Download the models (pinned to the commits recorded in the repository)

In [ ]:
env = {
    **os.environ,
    "MINERU_HOME": f"{WORK}/mineru",
    "MINERU_DOWNLOAD_MODELS": TIER,
    "MINERU_MODEL_SMALL_BACKEND": "torch",
    "MINERU_MODEL_SOURCE": "huggingface",
    "OTEL_SDK_DISABLED": "true",
}
done = subprocess.run([PY, "-m", "two_tier_parser.accurate.prepare_models"], cwd=REPO_DIR, env=env,
                      text=True, capture_output=True)
print((done.stdout + done.stderr)[-800:])
assert done.returncode == 0, "model preparation failed"

## 5. Start the service and wait for it

In [ ]:
import time, urllib.request, urllib.error

service_env = {**env, "MINERU_MODEL_SOURCE": "local", "WORKERS": "1", "MAX_INFLIGHT_DOCUMENTS": "1",
               "UVICORN_TIMEOUT_KEEP_ALIVE": "600"}
log = open(f"{WORK}/service.log", "w")
server = subprocess.Popen(
    [PY, "-m", "uvicorn", "two_tier_parser.accurate.main:app", "--host", "127.0.0.1", "--port", "8005"],
    cwd=REPO_DIR, env=service_env, stdout=log, stderr=subprocess.STDOUT)
BASE = "http://127.0.0.1:8005"
deadline = time.time() + 300
while True:
    try:
        if urllib.request.urlopen(BASE + "/health", timeout=3).status == 200:
            break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    assert server.poll() is None, "the service exited: " + open(f"{WORK}/service.log").read()[-1500:]
    assert time.time() < deadline, "the service never became healthy"
    time.sleep(2)
print("service is up")

## 6. The checks

In [ ]:
import io, json, re, socket, threading
from pypdf import PdfReader, PdfWriter

def two_pages(source_pdf):
    writer = PdfWriter()
    for page in PdfReader(source_pdf).pages[:2]:
        writer.add_page(page)
    out = io.BytesIO()
    writer.write(out)
    return out.getvalue()

def upload(base, path, pdf, name="doc.pdf", timeout=600):
    boundary = "colabcheck"
    body = (f'--{boundary}\r\nContent-Disposition: form-data; name="file"; filename="{name}"\r\n'
            "Content-Type: application/pdf\r\n\r\n").encode() + pdf + f"\r\n--{boundary}--\r\n".encode()
    request = urllib.request.Request(base + path, data=body, method="POST",
                                     headers={"Content-Type": f"multipart/form-data; boundary={boundary}"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return response.status, {k.lower(): v for k, v in response.headers.items()}, json.loads(response.read())
    except urllib.error.HTTPError as error:
        return error.code, {k.lower(): v for k, v in error.headers.items()}, json.loads(error.read() or "null")

def get_json(base, path):
    with urllib.request.urlopen(base + path, timeout=30) as response:
        return response.status, json.loads(response.read())

def run_checks(base, source_pdf, expect_gpu, log_text=None):
    """Run every check against a running accurate service; return {name: (passed, detail)}."""
    results = {}

    def check(name):
        def wrap(fn):
            try:
                detail = fn() or ""
                results[name] = (True, detail)
            except Exception as error:  # a failed check is a result, not a crash
                results[name] = (False, f"{type(error).__name__}: {error}"[:300])
            return fn
        return wrap

    pdf = two_pages(source_pdf)

    @check("health reports the expected device")
    def _():
        status, body = get_json(base, "/health")
        assert status == 200 and body["status"] == "healthy", body
        assert body["gpu_available"] is expect_gpu, f"gpu_available={body['gpu_available']}, expected {expect_gpu}"
        return str(body)

    @check("a two-page parse returns Markdown from MinerU 4")
    def _():
        status, _, body = upload(base, "/parse", pdf)
        assert status == 200, (status, body)
        assert body["markdown"].strip(), "empty Markdown"
        assert body["metadata"]["pages"] == 2, body["metadata"]
        assert body["metadata"]["version"].startswith("4."), body["metadata"]["version"]
        assert body["metadata"]["gpu_used"] is expect_gpu, f"gpu_used={body['metadata']['gpu_used']}"
        return f"{len(body['markdown'])} characters in {body['metadata']['processing_time_ms']} ms"

    @check("a second document is refused with 503 and Retry-After while one is parsing")
    def _():
        first = {}
        worker = threading.Thread(target=lambda: first.update(result=upload(base, "/parse", pdf)))
        worker.start()
        time.sleep(1.5)
        status, headers, body = upload(base, "/parse", pdf)
        health_status, _ = get_json(base, "/health")
        worker.join(600)
        assert status == 503 and headers.get("retry-after") == "5" and body == {"detail": "document_admission_saturated"}, (status, headers, body)
        assert health_status == 200, "health was not available during the parse"
        assert first["result"][0] == 200, first["result"][0]

    @check("an oversized declared upload is refused with 413 without being read")
    def _():
        host, port = base.split("//")[1].split(":")
        with socket.create_connection((host, int(port)), timeout=30) as connection:
            connection.sendall(b"POST /parse HTTP/1.1\r\nHost: x\r\nContent-Type: multipart/form-data; boundary=x\r\n"
                               b"Content-Length: " + str(600 * 1024 * 1024).encode() + b"\r\nConnection: close\r\n\r\n")
            line = connection.makefile("rb").readline()
        assert b" 413 " in line, line

    @check("every log line after startup is JSON")
    def _():
        if log_text is None:
            return "skipped (no log supplied)"
        lines = [l for l in log_text.splitlines() if l.strip()]
        started = next(i for i, l in enumerate(lines) if "Application startup" in l)
        native = re.compile(r"\[[A-Z]:onnxruntime:|^Object \d+ \d+ not defined")
        bad = [l[:120] for l in lines[started:] if not native.search(l) and not l.startswith("{")]
        assert not bad, bad[:3]

    return results

def report(results):
    width = max(len(name) for name in results)
    for name, (passed, detail) in results.items():
        print(("PASS" if passed else "FAIL"), name.ljust(width), "-", detail)
    failed = [name for name, (passed, _) in results.items() if not passed]
    print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
    return failed

In [ ]:
results = run_checks(BASE, f"{REPO_DIR}/examples/data/sample.pdf", expect_gpu=True,
                     log_text=open(f"{WORK}/service.log").read())
failed = report(results)
print("\nGPU memory in use:", sh("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader", check=False))

## 7. Result

In [ ]:
server.terminate()
assert not failed, f"{len(failed)} check(s) failed: {failed}"
print("All checks passed on", gpu, "with branch", BRANCH)

## Optional: the vLLM engine (`TIER = "standard"`)

The production image serves the vision-language model through vLLM, which this notebook does
not install (it is large and pins its own torch). To try it, set `TIER = "standard"`, install
`vllm` into the venv with the version in `deploy/Dockerfile.accurate`'s base image tag, and set
`MINERU_MODEL_VLM_ENGINE=vllm` and `MINERU_MODEL_SMALL_BACKEND=torch` in `env` before step 4.
A failure there says more about Colab's vLLM support than about the service, so run the
container on a real T4 host for the final word.